# Lucy’s missing digits
## A guided document-processing lab · ITAM · Tuesday 27 October 2026

**Your mission:** help Lucy decide whether her assistant has enough evidence to recommend a supplier contact for Saturday’s strawberry delivery.

You will investigate a real class failure, write two small Python functions, run controlled comparisons, and prepare a defensible memory payload. You do not need to have attended the live class: each mechanism is explained here before you use it.

**Prerequisites:** Python variables, dictionaries, lists, functions, and `if` statements. No OCR experience, GPU, API keys, or paid services are needed for the core lab.

**The story is fictional.** `55 0100` is a short classroom identifier, not a dialable phone number. Our six-digit format is an exercise rule, not telephone-number validation.

### Start in Colab
1. Upload this `.ipynb` to Google Colab and save your own copy.
2. Connect to a standard Python CPU runtime. Run the setup cell below.
3. Work downwards one cell at a time. Stop at **Predict**, **Try**, and **Checkpoint** prompts. Running everything at once skips the learning.
4. Edit the answer cells and rerun their checks. Hints and worked explanations are optional; opening one is not a failure.
5. At the end, export your lab report and save the notebook with your changes.

**What runs here?** The core uses embedded recorded results from the supplied class notebook, and computes normalization, experiments on those results, and policy decisions locally. It does **not** rerun Docling, PaddleOCR, Jev, or the hosted agent by default. New practice cases and schematic diagrams are labeled. Optional live OCR and original-fixture extensions follow the 90-minute lab.

The original notebook points to an unfinished `OWNER/REPO/COMMIT` download URL. Its 36-file bundle and page images were not attached. This edition removes that blocker from the core lab; it does not pretend those original images are embedded. The full original notebook is preserved in Appendix B.


### Your 90-minute route

Times are targets for thinking, coding, and revision—not measured execution times. Take longer if you need it. Optional extensions are extra.

| Stop | Minutes | You produce | Connection to the class |
|---|---:|---|---|
| 0 · Set up your casebook | 5 | A working notebook and an initial prediction | Lucy’s problem |
| 1 · Follow a field | 10 | Observation versus hypothesis | Page → region → reader → field |
| 2 · Choose a route | 12 | A tested routing function | Text layer versus page image |
| 3 · Normalize without inventing | 10 | Raw/value pairs and boundary tests | `0` versus `O` |
| 4 · Design a repair experiment | 13 | A controlled comparison and its limits | Crop versus scale |
| 5 · Turn evidence into a decision | 15 | A policy that preserves unknowns | Candidate versus accepted fact |
| 6 · Independent supplier case | 15 | A corrected policy and two separate decisions | Ready for Thursday |
| 7 · Explain and export | 10 | A report with provenance and a next action | Back to Lucy |
| **Total** | **90** | | |

**If you get stuck:** spend two minutes describing the input and expected output, open the hint, then compare the worked explanation. A reference implementation lets you continue without erasing your attempt. Passing a check means those cases passed; it does not prove a production system is safe.


In [ ]:
# 0 · Setup — standard Python + Colab's built-in display support. No installs.
import sys, json, re, copy, hashlib, html, base64, zlib
from pathlib import Path
from collections import Counter
from IPython.display import display, HTML, SVG

LAB = Path('lucy_document_lab')
LAB.mkdir(exist_ok=True)
answers = {}                    # Your written reasoning lives here.
check_results = {}              # Checks are feedback, not a grade.

def table(headers, rows):
    esc = lambda x: html.escape(str(x))
    head = ''.join('<th style="padding:10px;text-align:left">'+esc(h)+'</th>' for h in headers)
    body = ''.join('<tr>'+''.join('<td style="padding:10px;border-bottom:1px solid #ddd">'+esc(v)+'</td>' for v in row)+'</tr>' for row in rows)
    display(HTML('<div style="overflow:auto"><table style="border-collapse:collapse;width:100%"><thead style="background:#edf4f7;color:#17283b"><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table></div>'))

def check_function(name, fn, cases):
    passed = 0
    for value, expected in cases:
        try:
            actual = fn(copy.deepcopy(value))
            ok = actual == expected
            print(('PASS' if ok else 'TRY AGAIN'), repr(value), '→', repr(actual), '| expected:', repr(expected))
            passed += ok
        except Exception as e:
            print('TRY AGAIN:', type(e).__name__, str(e))
    check_results[name] = {'passed': passed, 'total': len(cases)}
    print(f'{passed}/{len(cases)} checks passed. Revise and rerun if needed.')

print('Ready. Python', sys.version.split()[0], '| core lab: no downloads or API calls')


## 1 · Follow the field — 10 minutes

Lucy sees a delivery note. Her assistant sees the output of several transformations. A **text layer** is text already stored in a PDF. **OCR** converts page pixels into characters. **Layout analysis** groups page regions. **Extraction** associates text with a named field such as `supplier_contact`.

Each stage can lose information or attach it to the wrong thing. A page-level score summarizes a tool-specific signal; it does not tell us whether the particular digits Lucy needs survived. Different score types should not be treated as interchangeable probabilities.

### Predict before revealing the evidence
Suppose the OCR score is **0.986**. Would you use the contact without inspecting it? Write a sentence explaining what you would check first.


In [ ]:
answers['initial_prediction'] = ''  # Your sentence; keep it even if you change your mind.
print('Prediction saved.' if answers['initial_prediction'].strip() else 'Write your prediction above before continuing.')


### Inspect the class evidence
These values are transcribed from the **saved outputs** in the uploaded notebook, not generated by a parser in this session. The supplied notebook’s live-render cell 5 reports **0.986**; its historical replay in cell 25 reports **0.985**. Keep the run labels separate.

The diagram is a schematic of the reported records, **not the original page image or a spatially faithful overlay**. The reported final layout box is not a capture of the OCR input region.


In [ ]:
# RECORDED class evidence, represented as data so we can inspect it.
TRACE = {
    'mild': {'box': [77, 210, 298, 238], 'layout_confidence': 0.76, 'text': 'Tel. 55 0100'},
    'faint': {'box': [77, 209, 210, 240], 'layout_confidence': 0.685, 'text': 'Tel. 55'},
}
table(['Case', 'Final box [left, top, right, bottom]', 'Text attached to box'],
      [(name, row['box'], row['text']) for name, row in TRACE.items()])
display(SVG('<svg xmlns="http://www.w3.org/2000/svg" width="900" height="200" viewBox="0 0 900 200"><rect width="900" height="200" fill="#f5f1e8"/><text x="25" y="30" font-family="sans-serif" font-size="17" fill="#334155">SCHEMATIC · reported final records, not the source scan</text><rect x="25" y="55" width="390" height="115" rx="12" fill="#ffffff" stroke="#277c73"/><text x="45" y="87" font-family="sans-serif" font-size="20">Mild scan · recovered text</text><text x="45" y="133" font-family="monospace" font-size="30">Tel. 55 0100</text><rect x="445" y="55" width="430" height="115" rx="12" fill="#ffffff" stroke="#b65b35"/><text x="465" y="87" font-family="sans-serif" font-size="20">Faint scan · recovered text</text><text x="465" y="133" font-family="monospace" font-size="30">Tel. 55</text><text x="650" y="133" font-family="sans-serif" font-size="19" fill="#b65b35">four digits missing</text></svg>'))


### Try: separate what you know from what you suspect
Classify each statement as `observation`, `hypothesis`, or `unsupported`:

1. The text attached to the faint scan’s final box is incomplete.
2. The OCR reader was definitely never given the last four digits.
3. Region selection may have contributed to the failure.
4. A second tool must recover the correct number.

Then explain why comparing **final output** with **input to a stage** matters.


In [ ]:
answers['evidence_labels'] = ['', '', '', '']
answers['stage_explanation'] = ''
expected = ['observation', 'unsupported', 'hypothesis', 'unsupported']
for i, (given, want) in enumerate(zip(answers['evidence_labels'], expected), 1):
    print(i, 'PASS' if given.strip().lower() == want else 'Revisit the evidence: expected '+want)


<details><summary>Worked explanation — compare after committing your answer</summary>

The final record establishes what survived into that record. It does not reconstruct every earlier stage. Region exclusion is a plausible explanation, but recognition or later merging could also contribute. An intervention can test a repair without proving the original failure’s cause. **Checkpoint:** you can say both “the field is incomplete” and “the exact cause remains uncertain.”

</details>


## 2 · Choose a reading route — 12 minutes

A PDF can contain useful text, an image, or both. Counting extracted characters is a cheap diagnostic. Zero characters means this extraction route yielded no text; it does not prove the file contains only an image. Nonzero characters do not prove that the text matches the page.

The class contains a deliberately incorrect embedded text layer: the page identifier is `55 0100`, but text extraction returns `55 O1OO`. The `O` characters are letters. Both this file and the clean born-digital file received parse scores of 1.0 in the recorded run.

### Worked example
For the wrong-text file, a format check rejects the letters. A sensible next action is to compare the text layer with a reading of the page image. Blindly replacing every `O` with `0` would invent a correction without evidence.


In [ ]:
ROUTE_CASES = {
    'born-digital': {'characters': 111, 'shape_ok': True, 'conflict': False},
    'embedded-wrong': {'characters': 202, 'shape_ok': False, 'conflict': False},
    'scan': {'characters': 0, 'shape_ok': None, 'conflict': False},
}
table(['File', 'Characters from text-only extraction', 'Recorded text / finding'], [
    ['born-digital', 111, 'Tel. 55 0100; parse_score 1.0'],
    ['embedded-wrong', 202, 'Tel. 55 O1OO; parse_score 1.0'],
    ['scan', 0, 'No text returned by this route'],
])


### Try: write `choose_route`
Return one of `ocr`, `check_text_layer`, or `text_layer` using this **teaching policy**:

1. No extracted characters → `ocr`.
2. Otherwise, an invalid/unknown field shape or an observed conflict → `check_text_layer`.
3. Otherwise → `text_layer` as the first route, **not automatic acceptance**.

Use `.get()` for optional dictionary fields. Spend four minutes on your function before opening the hint.


In [ ]:
def choose_route(facts):
    # TODO: replace this return with the three decisions above.
    return None

route_checks = [
    (ROUTE_CASES['scan'], 'ocr'),
    (ROUTE_CASES['embedded-wrong'], 'check_text_layer'),
    (ROUTE_CASES['born-digital'], 'text_layer'),
    ({'characters': 40, 'shape_ok': True, 'conflict': True}, 'check_text_layer'),
    ({'characters': 40}, 'check_text_layer'),
]
check_function('routing', choose_route, route_checks)


<details><summary>Hint — open after trying</summary>

Start with `if facts.get("characters", 0) == 0`. To distinguish an explicit success from an unknown field, use `facts.get("shape_ok") is not True`. A conflict takes priority over a valid-looking shape.

</details>


<details><summary>Worked explanation — compare after committing your answer</summary>

A possible implementation:
```python
def choose_route(facts):
    if facts.get('characters', 0) == 0:
        return 'ocr'
    if facts.get('shape_ok') is not True or facts.get('conflict', False):
        return 'check_text_layer'
    return 'text_layer'
```
This is a starting policy, not a general-purpose PDF classifier. A valid-looking but incorrect digit can still pass the shape check. The route chooses what evidence to gather next.

</details>


### Connect to Jev
The class sent typed route choices to Jev. A typed choice constrains the answer’s structure; it does not guarantee the chosen route is appropriate. Below are the **saved recommendations**, not fresh calls. The original route loop printed a recommendation; it did not dispatch a parser.


In [ ]:
JEV_ROUTES = [('born-digital', 'text_layer', 0.88), ('embedded-wrong', 'text_layer', 0.61), ('scan', 'ocr', 0.64)]
table(['File', 'RECORDED Jev recommendation', 'Uncalibrated score', 'Your route'],
      [(n, r, p, choose_route(ROUTE_CASES[n])) for n, r, p in JEV_ROUTES])
answers['route_disagreement'] = ''  # Explain the embedded-wrong disagreement without relying only on its score.


**Checkpoint:** identify the evidence that makes Jev’s middle recommendation unsuitable. A lower score alone does not diagnose the wrong text layer. Changing a confidence threshold is not a substitute for checking the characters against the source.


## 3 · Normalize without inventing — 10 minutes

**Raw output** is what the reader returned. **Normalized value** is the result of an explicit formatting rule. Keep both: otherwise, you cannot tell whether an error came from the reader or your code.

Our exercise normalizer accepts one standalone identifier containing exactly two digits followed by four digits, with optional space between the groups. It tolerates `Tel.` and terminal punctuation. It rejects missing digits, letters inside the identifier, an identifier embedded in a longer word/number, and multiple candidates.

This is deliberately narrower than general phone parsing. It never changes `O` to `0`. It must also reject `15501007`, rather than silently extracting six digits from the middle.

### Try
Use the supplied regular expression to find candidates. Return the normalized string when there is **exactly one** match; otherwise return `None`.


In [ ]:
IDENTIFIER = re.compile(r'(?<![A-Za-z0-9])([0-9]{2})\s*([0-9]{4})(?![A-Za-z0-9])')

def normalize_identifier(raw):
    if not isinstance(raw, str):
        return None
    matches = IDENTIFIER.findall(raw)
    # TODO: require exactly one match and join its two groups with a space.
    return None

normalization_checks = [
    ('Tel. 55 0100_', '55 0100'), ('Tel.550100', '55 0100'),
    ('Tel. 55', None), ('Tel. 55 O1OO', None),
    ('15501007', None), (None, None),
    ('55 0100 or 55 0200', None), ('refABC550100', None),
]
check_function('normalization', normalize_identifier, normalization_checks)


<details><summary>Hint — open after trying</summary>

`findall()` returns a list of pairs. If there is one pair, `" ".join(matches[0])` produces the normalized identifier. Returning `None` is a useful result: it says this rule did not establish one complete candidate.

</details>


<details><summary>Worked explanation — compare after committing your answer</summary>

Add:
```python
if len(matches) != 1:
    return None
return ' '.join(matches[0])
```
The underscore is removed by selecting the matched digits, not by changing the raw record. A perfectly shaped identifier may still belong to the wrong person or be an OCR mistake. **Checkpoint:** formatting and factual correctness are different checks.

</details>


In [ ]:
# Reference bridge: later sections remain usable if your exercise is unfinished.
# This does not replace your function or count its tests as passed.
def reference_normalize(raw):
    if not isinstance(raw, str):
        return None
    found = IDENTIFIER.findall(raw)
    return ' '.join(found[0]) if len(found) == 1 else None

print('Later policy examples use the disclosed reference normalizer.')
READS = {'tesseract': 'Tel.:S5', 'docling': 'Tel. 55',
         'granite_docling': 'Tel. 55', 'mineru_flash': 'Tel.55 0100'}
table(['Reader', 'RECORDED raw output', 'Locally normalized'],
      [(reader, raw, reference_normalize(raw)) for reader, raw in READS.items()])


## 4 · Design a repair experiment — 13 minutes

One reader recovered a complete candidate. Can a targeted reread help? Three distinct interventions are possible:

- **Crop:** change the area sent to the reader.
- **Scale:** change the pixel dimensions.
- **Reading mode:** change how the OCR tool expects text to be arranged.

If you change all three and the output improves, you cannot isolate the contribution of any one change. Before looking at results, choose a pair that investigates cropping while keeping the other listed factors constant.

The experiments below are Tesseract experiments. Their success cannot establish the cause of a different tool’s failure inside Docling.


In [ ]:
EXPERIMENTS = [
    {'id': 'A', 'region': 'whole', 'format': 'original JPG', 'scale': 1, 'mode': 4, 'raw': 'Tel.:S5'},
    {'id': 'B', 'region': 'whole', 'format': 'grayscale PNG', 'scale': 1, 'mode': 4, 'raw': 'Tel.:S5'},
    {'id': 'C', 'region': 'crop', 'format': 'grayscale PNG', 'scale': 1, 'mode': 4, 'raw': 'Tel. 55 0100.'},
    {'id': 'D', 'region': 'crop', 'format': 'grayscale PNG', 'scale': 1, 'mode': 7, 'raw': 'Tel. 55 0100.'},
    {'id': 'E', 'region': 'whole', 'format': 'grayscale PNG', 'scale': 3, 'mode': 4, 'raw': 'Tels--5S |'},
    {'id': 'F', 'region': 'crop', 'format': 'grayscale PNG', 'scale': 3, 'mode': 7, 'raw': 'Tel. 55 0100_'},
]
table(['ID', 'Region', 'Input preparation', 'Scale', 'Mode'],
      [(e['id'], e['region'], e['format'], e['scale'], e['mode']) for e in EXPERIMENTS])
answers['crop_pair'] = []           # e.g. ['A', 'B']; choose your own pair.
answers['crop_prediction'] = ''     # Predict and explain before the next cell.


In [ ]:
# Reveal RECORDED results from original cell 21; normalization runs locally.
table(['ID', 'Exact recorded output', 'Complete candidate'],
      [(e['id'], e['raw'], reference_normalize(e['raw'])) for e in EXPERIMENTS])
pair = [e for e in EXPERIMENTS if e['id'] in answers['crop_pair']]
if len(pair) == 2:
    changed = [k for k in ['region', 'format', 'scale', 'mode'] if pair[0][k] != pair[1][k]]
    print('Factors changed:', changed)
    print('Controlled crop comparison:', changed == ['region'])
else:
    print('Choose two different experiment IDs, then rerun this cell.')
answers['repair_conclusion'] = ''  # Observation + scope limit + next experiment.


<details><summary>Worked explanation — compare after committing your answer</summary>

Compare **B with C**: both are grayscale PNG, scale 1, mode 4. The crop recovered the identifier on this fixture. Compare B/E to investigate whole-page scaling, and C/D to investigate reading mode on the crop. These results support a repair for one page and one reader. They do not prove Docling’s OCR input excluded the digits. The crop was manually chosen with knowledge of where the line was; automatic localization is a separate problem.

</details>


### A deliberately bad “improvement”
The class also applied automatic contrast enhancement to the enlarged crop. The result was `Tei.:55 0100_`: the digits survived but the label got worse. A strict search for `Tel` could now miss a correct identifier.

Spend three minutes answering: **what would you measure if your task is (a) recognizing the identifier, versus (b) recovering every character of the line?** Explain why “the OCR got better” needs a specific target.


In [ ]:
answers['repair_measure'] = ''
raw_before, raw_after = 'Tel. 55 0100_', 'Tei.:55 0100_'
table(['Case', 'Raw', 'Identifier', 'Contains exact label Tel'], [
    ['Before contrast', raw_before, reference_normalize(raw_before), 'Tel' in raw_before],
    ['After contrast', raw_after, reference_normalize(raw_after), 'Tel' in raw_after],
])


### A fifth reader is not necessarily new corroboration
The supplied notebook later ran PaddleOCR. Its recorded faint-scan result was `(0.936, 'Tel. 55')`. Adding another incomplete reading does not give us a second **complete** identifier. Agreement on a prefix is not agreement on the whole field.


In [ ]:
# RECORDED PaddleOCR outputs, supplied notebook cell 28.
PADDLE_RECORDED = [('clean', 0.945, 'Tel. 55 0100'), ('mild', 0.951, 'Tel. 55 0100'),
                   ('faint', 0.936, 'Tel. 55'), ('heavy', None, None)]
table(['Case', 'Recorded line score', 'Recorded phone line'], PADDLE_RECORDED)
five_reads = {**READS, 'paddleocr': 'Tel. 55'}
complete = {k: reference_normalize(v) for k, v in five_reads.items() if reference_normalize(v)}
print('Complete candidates:', complete)
answers['fifth_reader'] = ''  # Why does this not establish the candidate as correct?


## 5 · Turn evidence into a decision — 15 minutes

The agent can organize evidence and suggest actions. The application must state what evidence it requires for the intended use. In the recorded class session, the hosted agent called a tool that returned saved parser strings. It did not read the images live, place a call, or write memory.

Read its original saved answer below. List one useful thing the agent did and one thing it did **not** establish.


In [ ]:
RECORDED_AGENT_ANSWER = '**CANDIDATE: `55 0100`** for `fn-spot`, returned whole by `mineru_flash`.\n\n- `docling` and `granite_docling` both returned only `55`, agreeing with the candidate’s prefix.\n- `tesseract` returned only `S5`, a partial result that does not exactly agree.\n- No other reader returned the whole candidate.\n'
print("RECORDED hosted answer — no API call in this notebook")
print(RECORDED_AGENT_ANSWER)
answers["agent_contribution"] = ""


### Our explicit tutorial policy
The function below is a **new, inspectable teaching implementation**, not the unavailable original `accept_phone.py`.

1. No completed processing → `not_yet_processed`.
2. Unreadable source → `unreadable`; we cannot infer absence.
3. No complete candidate → `unresolved`, with `not_found` or `incomplete` reason.
4. Different complete candidates or contradiction with a supplied record → `conflicting`.
5. Acceptance requires two distinct reader identifiers agreeing on the complete value and a source region from one of those agreeing readers.
6. A supplier-associated contact may be retained with role `unknown`. **Recommending it for ordering requires observed ordering-role support.**

Reader names do not prove statistical independence. Source regions are supplied evidence here, not automatically verified geometry. An existing record is optional and may itself be outdated. This conservative exercise policy is not a universal requirement to run two tools on every document.

We keep two outputs: `accepted_fact` and `contact_recommendation`. An accepted generic contact can exist without an ordering recommendation. Document ID, page, region, raw outputs, role, and limitations travel with the accepted value.


In [ ]:
def decide_contact(case):
    """Local teaching policy: evidence in; a fact and a recommendation out."""
    out = {'state': 'unresolved', 'reason': '', 'candidate': None,
           'accepted_fact': None, 'contact_recommendation': None, 'next_action': ''}
    def stop(state, reason, action):
        out.update(state=state, reason=reason, next_action=action)
        return out
    if not case.get('processed', False):
        return stop('not_yet_processed', 'No completed reading', 'Process the source')
    if case.get('readable') is not True:
        return stop('unreadable', 'Source readability unresolved', 'Ask for a clearer source')
    raw = case.get('reads', {})
    values = {reader: reference_normalize(text) for reader, text in raw.items()}
    found = {reader: value for reader, value in values.items() if value is not None}
    if not found:
        reason = 'incomplete' if any(raw.values()) else 'not_found'
        return stop('unresolved', reason, 'Inspect the source; do not infer absence')
    distinct = set(found.values())
    if len(distinct) != 1:
        return stop('conflicting', 'Complete readers disagree', 'Inspect source and reread')
    value = next(iter(distinct))
    out['candidate'] = value
    if case.get('record') is not None and case['record'] != value:
        return stop('conflicting', 'Existing record differs', 'Investigate; do not overwrite')
    agreeing = list(found)
    regions = {r: box for r, box in case.get('regions', {}).items()
               if r in agreeing and isinstance(box, dict) and box.get('page') and box.get('box')}
    if len(agreeing) < 2 or not regions or not case.get('document_id'):
        return stop('unresolved', 'Need corroboration and traceable source', 'Reread the relevant region')
    role = case.get('role', 'unknown')
    fact = {'value': value, 'document_id': case['document_id'], 'supplier': case.get('supplier'),
            'role': role, 'readers': agreeing, 'raw': raw, 'regions': regions,
            'limitations': ['Reader errors may be correlated', 'Region/role supplied as teaching evidence']}
    out.update(state='accepted', reason='Identifier evidence meets this policy', accepted_fact=fact)
    if role == 'ordering':
        out['contact_recommendation'] = {'value': value, 'purpose': 'ordering', 'document_id': case['document_id']}
        out['next_action'] = 'Prepare recommendation only; do not place a call'
    else:
        out['next_action'] = 'Confirm the ordering role before recommending this contact'
    return out


In [ ]:
# Practice cases combine recorded reader strings with explicitly supplied metadata.
MILD = {'document_id': 'class/fn-mild.jpg', 'supplier': 'Fresas del Norte',
        'processed': True, 'readable': True, 'role': 'unknown',
        'reads': {r: 'Tel. 55 0100' for r in ['docling', 'tesseract', 'granite', 'mineru_flash']},
        'regions': {'docling': {'page': 1, 'box': [77, 210, 298, 238]}}}
FAINT = {**copy.deepcopy(MILD), 'document_id': 'class/fn-spot.jpg', 'reads': dict(READS),
         'regions': {'docling': {'page': 1, 'box': [77, 209, 210, 240]}}}
HEAVY = {**copy.deepcopy(MILD), 'document_id': 'class/fn-heavy.jpg', 'readable': False,
         'reads': {r: None for r in READS}, 'regions': {}}

for label, case in [('Mild', MILD), ('Faint', FAINT), ('Heavy', HEAVY)]:
    result = decide_contact(case)
    print(label, '→', result['state'], '| ordering recommendation:', result['contact_recommendation'])
    print('  Next:', result['next_action'])


### Try: change one thing at a time
Make a copy of `FAINT` in the next cell. Investigate these changes separately:

1. Add a second complete reader result. Why might acceptance still fail?
2. Supply a region **from a reader that returned the complete value**. What changes?
3. Change the observed role from unknown to ordering. What changes now?
4. Supply a conflicting stored record. Does the application silently overwrite it?

All added evidence here is **hypothetical exercise input**, not a new recorded run. Never promote these edits to claims about the class tools.


In [ ]:
trial = copy.deepcopy(FAINT)
# TODO: edit one field at a time, rerun, and record the change below.
print(json.dumps(decide_contact(trial), indent=2))
answers['policy_experiments'] = ''  # Explain acceptance versus ordering recommendation.


<details><summary>Hint — open after trying</summary>

In FAINT, Docling has a box but only an incomplete reading. Adding a complete Tesseract value still leaves no region attached to a complete reader. You may supply a hypothetical Tesseract region for the exercise; label it as supplied, not measured.

</details>


<details><summary>Worked explanation — compare after committing your answer</summary>

Corroboration and region support are associated with the **same value**. A region from an incomplete reader does not explain where another reader obtained the missing digits. After a complete second reading and supporting region, the generic contact can be accepted. With role unknown, no ordering recommendation follows. A conflicting stored value sends the case to investigation, because either record could be wrong.

</details>


### Check the policy’s boundary cases
Run the following instructor-supplied tests. They exercise missing data, disagreement, a wrong contact role, and a misleading source region—not merely the happy path. Read each test label and predict its result before running.


In [ ]:
def policy_checks():
    cases = []
    c = copy.deepcopy(MILD); c['role'] = 'ordering'
    cases.append(('ordering supported', c, 'accepted', True))
    c = copy.deepcopy(MILD); c['role'] = 'billing'
    cases.append(('billing must not become ordering', c, 'accepted', False))
    c = copy.deepcopy(MILD); c['reads']['tesseract'] = 'Tel. 55 0200'
    cases.append(('complete disagreement', c, 'conflicting', False))
    c = copy.deepcopy(MILD); c['record'] = '55 0300'
    cases.append(('stored record differs', c, 'conflicting', False))
    c = copy.deepcopy(MILD); c['reads'] = {'docling': None, 'tesseract': None}
    cases.append(('not found is not absence', c, 'unresolved', False))
    c = copy.deepcopy(FAINT); c['reads']['tesseract'] = 'Tel. 55 0100'
    cases.append(('region from incomplete reader', c, 'unresolved', False))
    cases.append(('unreadable', HEAVY, 'unreadable', False))
    cases.append(('not processed', {}, 'not_yet_processed', False))
    for name, case, state, recommend in cases:
        actual = decide_contact(case)
        assert actual['state'] == state, name
        assert (actual['contact_recommendation'] is not None) == recommend, name
        print('PASS:', name)
policy_checks()


## 6 · Independent case: the readable wrong contact — 15 minutes

**New fictional exercise, not a recorded parser run.** Lucy receives a different supplier’s note. The billing contact is clear. The ordering contact is damaged. A colleague proposes: “Use whichever number is readable so Lucy can confirm Saturday’s delivery.”

Your task has two parts:

1. Repair the colleague’s recommendation function and pass its checks.
2. Prepare a memory payload that retains supported evidence without substituting billing for ordering. Write a two-sentence message to Lucy explaining what is known and what she needs next.

Work independently for eight minutes before opening the explanation. The function must also handle a changed identifier; do not hard-code the expected number.


In [ ]:
PRACTICE_BILLING = {'document_id': 'practice/supplier-b/page-1', 'supplier': 'Huerto del Valle',
    'processed': True, 'readable': True, 'role': 'billing',
    'reads': {'reader_a': 'Billing: 55 0645', 'reader_b': 'Billing: 55 0645'},
    'regions': {'reader_a': {'page': 1, 'box': [20, 40, 210, 70]}}}
PRACTICE_ORDERING = {'document_id': 'practice/supplier-b/page-1', 'supplier': 'Huerto del Valle',
    'processed': True, 'readable': True, 'role': 'ordering',
    'reads': {'reader_a': 'Ordering: 55 0', 'reader_b': 'Ordering: 55 @'},
    'regions': {'reader_a': {'page': 1, 'box': [20, 80, 210, 110]}}}

def recommend_for_ordering(case):
    # BUG: replace the colleague's logic with a role-aware decision.
    fact = decide_contact(case)['accepted_fact']
    return fact['value'] if fact else None

changed = copy.deepcopy(PRACTICE_BILLING)
changed['role'] = 'ordering'
changed['reads'] = {'reader_a': 'Ordering: 55 0777', 'reader_b': 'Ordering: 55 0777'}
check_function('independent_recommendation', recommend_for_ordering, [
    (PRACTICE_BILLING, None), (PRACTICE_ORDERING, None), (changed, '55 0777'), ({}, None)])


In [ ]:
# TODO: complete this payload using decide_contact(...), preserving the two roles.
student_payload = {'accepted': [], 'unresolved': []}
answers['message_to_lucy'] = ''
answers['why_no_substitution'] = ''

def check_payload(payload):
    accepted = payload.get('accepted', [])
    unresolved = payload.get('unresolved', [])
    billing_ok = any(x.get('value') == '55 0645' and x.get('role') == 'billing'
                     and x.get('document_id') == PRACTICE_BILLING['document_id'] for x in accepted)
    ordering_ok = any(x.get('role') == 'ordering' and x.get('next_action') for x in unresolved)
    no_false_ordering = not any(x.get('role') == 'ordering' for x in accepted)
    print('Supported billing fact:', billing_ok)
    print('Unresolved ordering field + next action:', ordering_ok)
    print('No unsupported ordering fact:', no_false_ordering)
    check_results['payload'] = {'passed': sum([billing_ok, ordering_ok, no_false_ordering]), 'total': 3}
check_payload(student_payload)


<details><summary>Hint — open after trying</summary>

The policy already separates `accepted_fact` from `contact_recommendation`. For a recommendation, use the latter. An unresolved payload entry needs its role, original reader outputs, and a next action; it is not an accepted fact with a warning attached.

</details>


<details><summary>Worked explanation — compare after committing your answer</summary>

One recommendation implementation:
```python
def recommend_for_ordering(case):
    recommendation = decide_contact(case)['contact_recommendation']
    return recommendation['value'] if recommendation else None
```
One payload construction:
```python
billing = decide_contact(PRACTICE_BILLING)
ordering = decide_contact(PRACTICE_ORDERING)
student_payload = {
    'accepted': [billing['accepted_fact']] if billing['accepted_fact'] else [],
    'unresolved': [{
        'role': 'ordering', 'document_id': PRACTICE_ORDERING['document_id'],
        'state': ordering['state'], 'raw': PRACTICE_ORDERING['reads'],
        'next_action': ordering['next_action'],
    }],
}
```
Message to Lucy: “The note supports billing identifier 55 0645, but its ordering identifier is incomplete. Please obtain a clearer image of the ordering line or confirmation of the correct ordering contact.” No call or delivery confirmation has occurred.

</details>


### Transfer challenge: what if the number looks perfect?
Change the two practice readers so they agree on a complete but **incorrect** identifier, and keep a region attached. The policy may accept it because it lacks an independent ground-truth check. Explain this limitation. What additional evidence would you seek before a consequential action?

Do not call this a fix by simply adding a third tool. Consider whether that tool would provide new evidence or repeat the same failure.


In [ ]:
answers['shared_error_limit'] = ''
# Optional scratch area: construct the counterexample and inspect decide_contact(...).


## 7 · Explain, check, and export — 10 minutes

### Exit ticket
Answer without reopening the worked explanations:

1. Why can a high page score coexist with a missing field?
2. What does the crop experiment establish, and what does it leave unresolved?
3. How do raw text, a candidate, an accepted fact, and an ordering recommendation differ?
4. Why is “not found” different from “not present”?
5. Thursday introduces a cancellation clause and two contacts. What can be decided if the terms are readable but the ordering contact is not?

Thursday’s cancellation answer and contact recommendation may be blocked independently. Today’s notebook prepares a payload; it does not persist a database record or contact a supplier.


In [ ]:
answers['exit_ticket'] = {
    'page_score': '', 'crop_scope': '', 'four_representations': '',
    'absence': '', 'thursday_bridge': '',
}


### Self-assessment

| Skill | Ready when you can… |
|---|---|
| Trace | Name the observed failure without presenting a hypothesis as fact |
| Route | Explain the wrong-text case using evidence, not only a score |
| Extract | Reject partial/ambiguous values while preserving raw output |
| Experiment | Identify what changed and scope the conclusion |
| Decide | Keep billing evidence without recommending it for ordering |
| Communicate | Tell Lucy what is supported, what is missing, and the next action |

If one row is uncertain, revisit that section. The report below includes your answers and test status; it does not fill in answers for you. Empty fields remain visibly incomplete. Download the report from Colab’s Files sidebar and save your notebook copy.


In [ ]:
def empty_paths(obj, prefix='answers'):
    if isinstance(obj, dict):
        return [p for key, value in obj.items() for p in empty_paths(value, prefix+'.'+key)]
    if isinstance(obj, list):
        return [prefix] if not obj else [p for i, value in enumerate(obj) for p in empty_paths(value, prefix+f'[{i}]')]
    return [prefix] if isinstance(obj, str) and not obj.strip() else []

missing = empty_paths(answers)
report = {'lesson': 'Lucy — document processing', 'answers': answers,
          'checks': check_results, 'student_payload': student_payload,
          'incomplete_answers': missing,
          'evidence_mode': 'Recorded class outputs + local policy exercises; no hosted calls'}
report_path = LAB / 'my-lucy-lab-report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Report prepared:', report_path)
print('Unanswered items:', len(missing))
for path in missing:
    print(' -', path)
print('Save your notebook and download this JSON through the Files sidebar.')


## Optional extension A · Run a real OCR experiment — 20–30 extra minutes

The core lab is complete. This extension creates a **new synthetic image locally** and runs Tesseract if you opt in. It does not recreate the original class scan or guarantee the same failure. You may see both variants succeed; that is a valid result.

**Question:** on your new fixture, how does cropping affect the recovered text? Change only one factor at a time. Start with a clean image, then lower the contrast of the digits. Keep a log of contrast, crop, mode, raw result, and normalized result. Do not tune until you get a desired story and then conceal unsuccessful trials.

Set `RUN_LIVE_OCR = True` to run. Colab may need to install the Tesseract system program; this makes network requests but uses no API key. If installation is unavailable, keep working with the recorded experiment above. This optional path is not needed for any assessment.


In [ ]:
RUN_LIVE_OCR = False

if RUN_LIVE_OCR:
    import shutil, subprocess
    from PIL import Image, ImageDraw, ImageFont
    if shutil.which('tesseract') is None:
        subprocess.run(['apt-get', '-qq', 'update'], check=True)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'tesseract-ocr'], check=True)
    font_path = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
    font = ImageFont.truetype(font_path, 28) if Path(font_path).exists() else ImageFont.load_default()
    page = Image.new('RGB', (1000, 440), 'white')
    draw = ImageDraw.Draw(page)
    draw.text((40, 40), 'NEW PRACTICE NOTE - not the class scan', fill='black', font=font)
    draw.text((40, 120), 'Supplier: Huerto del Valle', fill='black', font=font)
    draw.text((40, 210), 'Tel. 55', fill='black', font=font)
    DIGIT_GRAY = 170             # 0 = black, 255 = white; change this later.
    draw.text((155, 210), '0777', fill=(DIGIT_GRAY,)*3, font=font)
    draw.text((40, 320), 'Strawberries: 10 kg', fill='black', font=font)
    page_path = LAB / 'new-practice-note.png'
    crop_path = LAB / 'new-practice-crop.png'
    page.save(page_path)
    page.crop((25, 190, 400, 265)).save(crop_path)
    display(page)
    live_rows = []
    for label, path in [('whole', page_path), ('crop', crop_path)]:
        run = subprocess.run(['tesseract', str(path), 'stdout', '--psm', '6', '-l', 'eng'],
                             capture_output=True, text=True, check=True)
        live_rows.append([label, run.stdout.strip(), reference_normalize(run.stdout)])
    print('LIVE on a NEW synthetic fixture; not a replay of class results')
    table(['Input', 'Raw OCR', 'Normalized'], live_rows)
else:
    print('Optional live OCR skipped. Core lab remains fully usable.')


## Appendix B · Original class code and optional original-fixture reproduction

The complete supplied 33-cell notebook—including its code, narrative, saved outputs, and metadata—is embedded below as a compressed archive. This preserves the original Docling, Tesseract, PaddleOCR, Unstructured, Jev, and hosted-session examples without making the core depend on missing helper files.

**Availability boundary:** the source `.ipynb` contains textual outputs but no original scan images, model files, or helper modules. Its original default download URL is unfinished. Extracting the archive does not supply those missing assets. Request the instructor’s original bundle to rerun the exact parser examples. Do not guess a repository URL.

The next cell extracts the original notebook locally without executing its code. You can download it or inspect a selected original cell below. All old “LIVE” labels describe that notebook’s recorded render; they do not mean a parser has run in this session.


In [ ]:
# @title Extract the original class notebook (no execution)
ORIGINAL_B64 = 'eJztfWtT40i24Pf5FRmu6Ciba1R+ya9adoIGVzd3qoAFanp6gHDLVhqrkS2PJAOeKiL24z4i9uNG7N+7v2TPOZkpZUoyYOii5869RBXYUj5OZp487zz55Q+sNOa+H5X67PwPjH2B/+LJMF4tODwtzZzw2g1u56UqvfNcfDiu1eT3GY8d14kdePrlXjyKgmU45rJF+P6GHZztfmJnSx65zoo1OuxoHAcjHrJGrdHus9spDzlzPZd9XI5XbyO2mAZzzubLGZZxvchZLLgT/vFCwsBK6aezqRexeRDzURBcs0ng+8FtxLD6zHH59nLBXO57NzxcUSkWT8NgeTVlDnOD8XLG5zFbeAsoMudVNgmDGZTgbOFcQdGAPk887rvMmTMnirwodqDGbbCER8uIW2xATYc8WvoxNDr2oRQbO6HLoilCMg5mPBINw1uYVxZMoFkN6CoLl3PZdbDAXkdBHAczaNsZT1XTUMF3Rtznbr9oGrbZ1tbHgz8Ptrb62OViGXOXjVY0AOp1FSyZA7PsQ4/e/Io5sWXWPhnsHZ3sD/axhZA7roI5cm6gqYnnc4QcpgFWwvdgXQBqi51B+/FtQH1E0JsTs9jxr3EUDpsGEYIxC2AJWPlf+Q1Udwkk+SbiMKXBvAIdLnxHQCv6c+bRLQ8jqhDBmyioshEfOzDnWAoWk3PRlJjxa76KrKKJGdx4Lp+PYeD8hgOI0IDLvDlVjfld3GdH358OTv482GflWwTfoeWgxatU2Y8/Hx+d/Tg4PThlZYctAoB3BDNBgFRxCRkHlBj5XjTlLpQ/ODzDxg7PDo4OscZ46swBlW4FOlZpPNTN2JlxNnLG15VCsHFeDcSNqgIx9Z1xUbJtVqvXahelZGZx8Z25BziHq42dMtgDkyAUSEezZRk14amD4w1j8TYMYN1x0mIPUD8Uo3RUn858hRCMARFcz/ELgd/aOgtoFr0Yp/qHILiCOdsLAIH7W1ssWHAx/+YOcABLBRqVxbYLo1jgrjeHORb4hVtzfA27U6BG7FzDpwiWNnR8NvPmgPdRxQJ0/pBUxy4BkEXozWPRxPEqhmlkUAmxT06s78QwSzNqltO0y57SchEiNe4NmAecgIhzsZiLMHCXY4mQwTKG7RcBugIpEjvE9UI+jjXQYWkW3nyuUNERkL6n+rK7CLcbtU747nKYNRcIG1UewwaCSYQNDsQL6aecazFPolOxMUKoBeAS0eFRCh3tA1xJfucAaBGPsUy9102hRKSZOeOjU+x1d7GANYw83xvDt7LACaw1WnpAnrAsLXBF4LgjvimQoB0YgjeZAChq8009IJIuH3szh4hiBJSSReMACJ5VQkS6hF/EUQqY0hhoimRA/I6PlzHM2HAcLOcxvK2bnKpewKkEoiZVk0dQK1gsR5Z4w4feHGYLm0FOtV2vbTc6ZzW7b7f7ra5VqzU6zdpfJdqrqjDgeBlZo2W0eqBis9dcU9FzfV5Y0a5Z7U6nXbfTikB1fD8BFqlocZ9Utd2udf9KM8vu8bfk1hIjEnatJmIOFAobi2IXiiRdiuLJQkQxbIhZ8hZJatISfEf0VDjA7wBHceH6rMbKDRZVEpKBc4AzHfVZs03MBtCPx+Mpl3wIH/XfvSPcgqfja+QRVw42Tdg0A5o3AVKst7gQ2xx+oFWr3oB/7KvAaJgZq2bVt51w1m7h7+12awTQlQlrQQIxSYSOvlJG8EjIMEYApCcCOgS7qc/Sz7ZlW3W9GMgdPnJg+QNgtRqFJbZxL4gStZZV00uEzsKDUqFsA0ZXMwvEQQiyA9M7MftYOC7gmfhNLTSL3iddNK2O2cESZj5cjuNlCCvBWM2CGW3aBgiAIZEz4duRe00lOpbxfuGhrJaAWG9amTEAx1ms9DE0zQaIacRifQsw/q9GWUEGkeP089RRLG2Uyn+JjMNhDVxcLaBaAh4CA2QFJ4KPttUCgqfWa+GEQGE7VgPmUpSgzzUoEocg0SB/wW5sq1G3aqIEfIa5qxBi7x1/RvkUGAzwwFaV4XviikgTK8R/iGpuK6opqKVkxjp2KjEKUJTEIjENl+nev1wnqdcsdsrj5aKfMijBVASbFLtU0XgxPZK5MD6/8cJgjjJ1Kgt4swVKFgHUiVb4azkCbjmG7VFNeC48nQIZ9qts6kRT3xtV2a8RMtxl6MM3K+R/A9UhrrK/ewskBFXmgTgYezOe9kMDXzgxVmey02P4WiSfvGE/kcaRDoihpGWxg7lkW7S/PSlwLEG8G6OIGkQe7KuVGi+h0HuYBj8Yk/QRhoEQDEYg2wD1JO6Oqs/w+93TgSYq4Ve2A5NiyTmzrnhcviglZS9KVZDRpnG8iIDqhc6tdeXF0yXwFB4C/41hii2Q9N4d/XQ4OHl3Mjg+erd39OnTwdk7JVNF7+IlfzdazmEXX5RSOlU6Pjg8hb7PLxSZ2dlRJIg61QnDzo6gC9oLoAj4FKlBqZq2qjbphUEYdnYUXaAWdIqAr5AgiKaJFOzsSBpQukwbjmsALC61hb/K2kBCeJFikwXUoHwOKCa5IcjkHJvenqkuFuKDxGfxZftv+HcL5+SyCku3QLCHgsXtnIVLaAI5Gn3UuiZZsnyxhrthkyEgLbQ1xwegV16Uyl+0QWzHtb5Vm9wjC9TXxpsYFftCaC2HIBoAmQrPt+t2rda/rLxnoeMBqTldgRo1G0DP5WRgbOLAFnGNdnXUP0s4Ewn+MH1XoTNTUr7ke1IGtEwpLkIh8xYQkcFEvWe0h0CzQZ0MJUmSY7FRH0kYiKZAmxBRjfGJnW7dTr3xFIBO+CTAi3UPoaG+iVbZNb4oQe/bsGHUEv6taGFXEuVU+9uAuIBYVSE9ZBc0/fTT0cmfALGQdJSL9yeWEI3fBuE1wF2BJYsC/wYW9j3Dt9bsGiT+MrADGH4k8QjwI4qHgez6PW7+8RSLYQ0NFERhpGMWjIeHcbkG1DGWpbRiHw4+DnAfI6G0/ABYRtmklhZ8RU2rTMTmXwDcd592Dw8+DE7PLKwEgFu4TuVKRWzdN+xLyIEiAyUkQtqHtXIadvteI7IBskQfOdOcVDuCwvIACaNyJbNuKGoDgI9CBSrov2CrChyzlStAyx3FFywBURmbrlhTfud6oKjE2ToOrnksqu4QrLQFcXj3alR5np+IkKXMOOAZgECI8Q4BfU+PLLG8T1prKn8bwjwNRytASTEAc14nOJ9l2csFKTi4RFd+MALM27IWgNGVvto/42CxKk+qCqqJhfymwrSfN8wZj/kiHpLJAKoDU+U3wLHpI2xuABekSPjG5kDilLWLNPAUsnQLwL77cPAX2EJIdiVCEqQT7w6pZgTgvWdG+aPPZ42CCkgWHBBP15AoybzFH1x0pbfBurKZO0xLunzCbsqLLOLF4QpNWEhFsbwlhXgoaJbjdzg/bEB/oEBS6aKExFDSEySlecovtRWkAz5gM+2DSpVmI6u7YBni/9HCJ0INGF85r10K7Bfon3/fuMTWHlV3SkVcKdV7sG8kKHIKZBfYO7b+FV8rIcxSH8o0js01oSJQNJWIQMmRco3+E9nelv0IUv0wP7aEZioGhQbciEaW2VfIoTVpR/Sja1jiidKoJNsICD+z8lBGEBJfdYEnL+foAo74TGoEjC+DtWLKkEwt+vXWPZZF3C6YVE3vwVIkVwAIE5ItLkrf/bz93Wz7O/fsux/7333qf3f616TU1UzIHxVT5ijmcyeDw/3ByfDw6GxAlCfpXlOlsGVjy+drXlYeNeas8TA0NvAwvAGODUSjbrEP5AJILffP7b359N7Ra0FsJkIqOkYh7AMQOSBXaPg+BDJGzgQQ3CdeOGOnDuCK66zeanQW1s+5HYGM54HCxk7ROMq5KxSQaLlYoDaadYwozqUTRM1ijFZA6WZAyco0BSsFGAmMF2uAAConHUAtaSYm3fOWJ1ZPMhr6K+m5UcZhDalAowqA573MjtcwV6T1Tex4ZBrr1RvtTe14VLHb63WeYcfr1FqdduNZdrxOrdnodl7bjjeZb4+CcL4NApcHXNFauBMlaNTbrUadkVzDlHDV5N2G63a563DDpAPNjH3uzK1FagCTP0270exlmnE7rtvrNDpuredmmuGAcK7L3e1bIDxXKTzNWg1w1mym026Obbs9HjXHjUwzU+7crKxfF1loWt223chC0671us1Ws1dv1jPNzDzfLWil3uh1es1MM06v1x13W7W6PXYyzUSLIC5upl3LNtMd89rYnnQdd2xAQxuY6ERf7d6NDEB1qy6JpxDr+qTboUMUiIB3l7gncgJbYrsJtHePm2VAqMzrW4mwqTGqs5PPZz9CUc13VSBBR0htXRDI/qJEZ5jXLWsL9bQMxwXON7Gi5QRGijW/XJSs+A4EkXt0oc5jb77kGU2AtI28OjIhzUWK9hVDMTnv1xuXZiuR93cObUyITpRRihnio3XSgBDt+43WPfuC5fqd+wwmfBGd3RdLYCk6ILOmSXycIz9IlpsmWba/FVnu1Dq23XgGWYaKMC+bkeVOv9G2Op16s9PZlCyLqu1up1V/HlkGtv9MsnwRphv/Y+CQpfqWe1dT9KQwVvvua0pFvrLau06nxs5rtX6t9l/+WGV/9OJ30aVJG6qvDWS7893Xf/u//yP3738TyHa9pQNNv6vMrrXqVrv1DwA+kKFi8NN/XxkMoGAQ7XbNtuzuKw1i/sQuXuDqQ185aEXDSPqt+qxm9bpt06sEZXxnBS2LYlimY5ueJ5JxUZ8TcSjQzvnbM+5bwM7eXm7IyxqiDWJooD7H28CLxlMR30OWzH3pjUNBGGNjmHC4S9dKjsn5wdUVFEcLWDiHD1oJ9cSaeH7MQ/UVjQVXc9IzK+9VfQuDmkY+L6vveycHZwd7ux8rGdYpFVVLBSsBEZ4DB45ROhcQ7cs3e+pF2gKWBU6TK6Hby9CGQnY3MiBlOCSa2LERS/ZaRjsOMux35N+oZAwq0nwSovkbZzgiO0OYAA+UFEEexsFQKV2GQX+I8k+VLeTfmUsfAAYC8EKXkAxeF42xDBW2SMmiiB8JQv0yzxTzqEo2e+BvbnniB8CWo7GVvK9UWbNID88jc0Er+vtsQzFohzvs3CedDMNc1JANkwbKKRcl2AAYqzNnftF4cluGVH7uv5DTt0xO3/4GnF7yXWDZGypgsmK706xvyumbLasLQny3vTmnx6rdXuf1AylQwwCi5MyZ/IGJfjsK7t5CmU6nyhp1YCmNXhd+Nbuw696mG+EtEdk2PkPj7luNnpIc/fby3tBDNDpJHWZ7qvVUd61aQU/trl3UVaYXpLHoUkVp3SDN0BPjczdKfPBImaeB76KZ0V8x2RzJ6BiPiRGcRMljCrQC9TRicwzgwq0wniJtF31R2FbZYdMV7LAph7pAj8lq1MJwjTgih/bC8WTIGxlJMM5vQ5bTlFGoqQmqTx8FJaDxOUQktIA+3Lo6IcTVRkIo/gJVoOXXCaHSOA1CiMScGhxCL2WkT1mKjpRmjGQEX1qLkLveGPdtJAmVNfaXMKdhlKmHP6D1DAXJcuarskaS7izEVmqbNKmxRatfyTeBDkBgIUgWR7CON46/5BV00lwQwss4RtlRAQj4M0KmZI1ghMXvJRcCdQ6L4NoIkjyy/IoizyMr1j6HlQJPdtGPqjCqXD5SAx04alcgDDpXGGtMiljC422RUwaHsmaqtb2VMIU8xSBzf4oehFRFjC1PAbI1kUkV1Xz5rs563+RUvnynFzX8pN1ferYBu7O5Abthsb1pEERcRprioAB3Yhk6jNFBJMn6aJp5Llzdp8P1PZ8EZOElYceLCCgRCpSGw8N+Jft2AjgCrLn9zwTU5BlzwhAdzMoKHWEYtIjKQjqINkYmbYxVfKKJ1xR3QO4fJKU8FPEG6I8agSyPVkCqgWGpgLMvND3b5oz1vonkQ3JIq9XubSr5iIqN5sY2jmbXqrfsZrf2DMkHqzY7jWdKPv+xbRzNYiPB/yGQW91mgXmg1W00rMZrmQe+kY2j1WvYVqPxT2Pj0H0g9KRer+MZD/RgYzCJjG1HUvcK0Jg+EHgCm/b3g0YXNeRP7YnQPChRN6w6SAq3GPOw0tvDcxAq1MzDY0kgZmDj2jGxrNEGo1BfyXDzoM+DShwffFRvD2bAXddZfoDd0Akqa+REfEgfo6Qi8pYPQThz4kerq2Nuw4CiXZI2jt3JsXx1JN78ljaoJ7YAQIhhCBCe4R4i9NsRU2lRiJkwVGW1pcSWdVE6+eH7jB0J5AYMnE5VrG16tnAnZNrBMD9i9jt1u4bRVm+EfIxx2ux4/4MWN7UAiXKnYG7LbjDEGNoPjo8nyOAbhaoOkxgS8cawEt3FQxKSi0x5E5o1taY7X0whV8MPC+DrZ+e5nEWKHQS8cl8QaHhevjDIH06INsNZ7zBMbJFaA42YVCvTTN6tu74hnHqsTshRtGKVimYrQ4kT6VyVtmYSvJgNWgzQlphMuWH4xHqVSoLEZkXEPYwEg7eamRMbIhdf6C3KWXNp6uMjD1/dvmdf5v3W/Tqi+QRF5EGZtm3ItPWiA7y/gUyLYmK33dvQb6cqttqbW/O6Vq9da9Wf4bfDqnbXbvyjhFPQARJplMYTIozpdrpX4OFrAivWw3VUPzrSzXojULhZUvA908KVUHlEiZIUWXwODwQJ2EwaaDB5TheB3ybCSEZvYfeRZ1PQNpB14aB1LOa+2JT4K2scIwO6aZDHYg9b489rGf++tD9h+XP8ZXlzl98pe1mlf5mJCfi4+/PgBLr9cp8nVjJ2sZC+VtdTzGxsYbiWpq135iD7Ch92p+Avgv4cG7ikTuT0Fjp9BDXM9kPIssOkx8TScO1hetlo3BuI+UX4YKz65B63hQZYcWTEs1D1pTS4Y9Lgb3E0VVLElkERn0yDoWKtu454r6XBrY5ld2vdbndzGoxVO1DT/k+7wuZ2hSLF/H8KgFvtQoXc7rWt5r9zqwIw7LrV+uexKpBNM2Vk+NNfy2GBizmYECQpqheWYkJx4dThXZ6D0jLjzhzDg9nR3kmFYjV6ZiChTO3hXIWc9xmpJZuy6lFq5KUMIkrvc43gi7cYKC0AjXgcm9o5ZuJYr9gl0aSmZvfRoPhvMp42dYBzsvR9EY5HQdLlO9atoSEbSGCVrVij3cBvzUavkgHHGofBolzuog+03aiKCliOjoJBg+VyvYlv620bngmV0uBlF6Ubj1KJbJOIIAag8X+E7JHIDWTqWK6An69XpUxnIU35Y1w7DdXQKgsL/I5k/+sFEU0jHocJly+UK7SAi6KDJPktQodvEI6C8kW7BMvL54/VeHCrpFEeUGxNbIi5eRJI0dWpQHiZINE1BYmisxIvFySIrbfa7Q1zXKiKzfaGsfGdvt21gIa17GeEZmDVWqeNcST/KUhsCGSjrnhxCmjd7hbw3rrdab9e1OKDQHe7DwkQ/wuG0O50CobQbPdsq/1a4sM3koGarV7Hqv3zRI+qrCgJ5aVQopYRParKiNRQpJBjgNGbN+zDyeB095TtDz6yw6OTs8HbKnuL/xu12GG1Nvxr3/R6jtVhNedr/abb6VCJXDTpNxHu1BHIDNhFMB+BjORMGL+O5xxU/44LvDn2uBOvQBWNORb5wL23l/qsaHLNhMKKRitlcJdSG6NMcAqOVJQzRL2IcgmwKA4WkTjaTzk+omuRCMphIxDNgIMJOW9DQbApsr7hYRuymPdlii0Zq3HF44hdOeGITpkAny2KipKegvQMafpKNCyOGWsSUHK4xwySKhasEjGKamnlZ1jBfWoQqzJnuA+LOAXyR24D5KNJHwhGLdob2MDMNc/H9lu602RMg8uew6UZQPOSQHx1NHcRyVwaLflE5lrgKLzqKTvyiTmgn6ce59WEYBpKxh6XM8b5yrp+ed5vFsTGFu8/hJy+a5MxWg3VLOZC3IwYXQOGosDgB3Ykdpx2lPNsQG/GXhUznIBRJLg+Y/M+O6SpvsGpWBXS1BTp6CjjAIWsBfM4DHyR5U8mgkzSPQLkKhHls2EsOie6BsZDGaaZxjIBtopTt5jcw2K7mVSVTGRnMk7d6nFupEiSbgsrH0r6FlkMk1/iwD59Ghzun+IhA/RXa2Qj5gthucZqKPmz3ePjjweDU6G7Br43Xq3L26hvGYrmdPwHUzkuyYVtHCbWorV+Qus6qNU3PBKn+2cUXMeVL0okfYIpwYPOwMmi1Rwex944bQL2NDmwXHY2+MsZkmXU6YX+VjZ18UiL2dMoujCiE2bI5KMTaDACugKT7ESpi4yRQoQ5Hw8D5qGNIMqORUSt+WnuTTwV7YShAwD2cWXeRiJkjYY2AyIPWGM4oakFOWpshbYed5WrAVPU0vJzUhLFYAjrNRZlohHNHa21MGET3kyWc1q+KqP0OMqjoVFWffqTWZ3IKRRZNQiD1Ur7IpkL9BAa8XgKo0RyCVzZTLIQnCOXw/zOvDnsDG8scFKflBhzUCootNYPBwPA8LMfB2zv4+7pKTvd/1PEdg/32Z8GP2MWVS3ziJ6SJG2AcrkuVpW+yi9G+0mQhZ16VSWLxacYsfwyPbpn0o5vcZhRarV2t93aVI8WFVu15+jRDTwo/oxAP6oK0D7iFF0r7WHyO4fh5LuMcv4gviMXgp3Wp0geIFqYP5mIC+3tkaAAWuSOyG0g82gMMYOgFP/2poE31vbVyWB3H8MTMoEPwmI0JKMLBk4jL5d+9CRoSIQfq0TAFvsM6AybGahLPlbY+LkoYbJVwPlcaMJFidKCJP2pqFmHLTyK7rDYCdqXVB6tdR1dlI72Tgrbpw07zA4u14g2XDlMqAGSxzLGvvWgWQlRkvW6CCRMYELgszgIRA5OTDMMwwEQNbflf/s8OKXMx7geJZxlBE8sWQZIT2ZNoaATmK3dTJ5uggUDi0XAsEsYJNODP7w8F2ijXVjs2APKRgl5gdXcYnZpQZpxAih0eIJO4sfaIoZxy9n1HBDXGeEJDhWQUbA+Y8zxFHrODqFl5f6FeWxrJnH6NuevaL+3m+2Nz1+JivXWxlHIWLHV7tit5xGnlm3bzzxp/dtFfqJAl4gLYnuJDSkxp2Z1u7qKnY3VfLx+2/DGZKMrM/XTXLBp/dZGSnpTBVaYyJ6EVYrgekpuAxsJuzclWKLYgp0rWg36Rz6OLEeozaCyPlKONAAK8HYogp/6GGv72Dkd0oNFiKEWGkCVrdrjlVNReYhnkG4c33OH0dRZUBOopd7nCHI2nO2BAQA2P37Q6EVDeHwQZPvJj0LE0j0Ae+2+IDwFUxM6MX8sE+AC9enoGkM3olpzm2TtbQrioDRX1EaVKeZROVesIxNqgrcj7ADvuiuDqgRC4s4C01U9HlSX2SpfoJ1+vX3PvizO4eNl32pM7l8c2FE3k47Xi87+/Da0utXr1J4jSELFVm+dJ+dBWt1t9nrPFCS7jXat99q0emtrD/SNg/3ds0Gf/SI15l+2tkiG+EWeIv+lyhIl8xZkJI6Gl19A3eHhcjgB7Xn6i6UTYP3zNvtFGnB+IWnol6vQmWN2x+QpBRcl7ZPiC4D8UhUOQXICqJNssPNcD1CE/9t//38RJk2deHdWprfEIPRLttFTahR1vNgDziRvCyH5xg1IH42TbF3Ud6Zp0JrxjJwyVaTNkw2E5iUBcLNM0k1l9c0aUJA3oEictCvmAqUxLm/ryLCVRGdUnMUHQXAoW9T02qOjj6cF9sxM8r/MYomH+sIblOd49+R0MMSmC7WMlSCrFyWlvMskf4DH4nmq3+cSJqPTOQKRUYS8C63hREQPZgIWHJlPOrtIjygqCBagPDZjStfl1EKTeNYrhfoGgA89IzeQ7CEdcTD6lasJXoTBggMKclmsQIUQU5BpQ5gPLkpZjiRnNwj8tVWE6Xk5I9aMK39f3AhmnvUoN2MfMUOuRNL6pc7YSGwffv9zut46JvWFKdbqn9oZtFJvmG0XpsXWFiSHfRvUNZFUVUxMghj3+IaFzq26yahlBr+mqIj3d6DxHWagkqQ9NQZ/ju+0bSDMtzQjdJxEdH+1iLfb2w5mLyyGXaRlVopegbJ6UfpZ3kwklTuYE2kzQIxy4iC02B5KnZqhLMALfRZ0A05uQwLYyelSOUq8HkWQHW1fpWKSOL6Rb0pjInQvCx5WR4+DQ/OWbkJU5pFQKrNdMVCRIL+uugMIS4rnJkVHSo42PS+22H5Q0BhS9KslYCWbeUAAgZWIs9AWw5l0A2H6I3Meo3zNMy5vC8q25MxXMR6bfk+3Q8noGpwm0dTUucEpFFYEBLWvWs83hXc0YSPq9ibRcAFF0Xa2ONGe0lf9EDvmON0xyHxZCI+Eh9uUkp4+PiwRIz3NrHvKdShnnEwp8/DOY4j2OiHva9h4n/PJ4M1YE2/u+NLV9VLR0syDWS86vv3biJbddre+7lqaB0XLbtvudZ8hWjbrDXj8LNGyWa91Gq9uBjB1CSSdQ7Ip1Sy7owtWQIQVsxYdiIQtCTN522dvJStBp75kBuoppkuBpxlWkXmrMwP1SmV0MVKtzPHAl+/hhU9Ew6I8LJh23wBDPshDIF9kOy/sWPoYpExKP32NOgE8X0GP4wtBgNONiZToPWqMRc5LRo8iWIO5kofeK0rYl2VgTa7IjwKqcxiMgtD05+DuSzpLfpJ0mQBV+lpJpDhj0ks1RHF1CBRlSGR/SJNKqW6MWcHUOEYNR4KFRc1kOMl8oJNLQXOopz/AQpgFeEwjYZRLPmJGPeFfwwneUExvZTTk96kPkvqUPjrsMwZOKlY1K6Fr7iMlpP+6dPVTrfuDvYP9gTING462pxqJKdHybIkXuwXzGOfKSbIjY+bn1OP6qF1X13O8kBku+4zE/FhTItnyiRpScmmb5lWWHg83z8+zjSVqYSqfPGRnLpS0xWJIheJoLi4F9HnMDV9wMgOg984waOVRrwdolXOaGunhLObuOLdDZfsny35WazHkHBzr42qMNhupv0KoRpgWfy0oQESGKEVIWA6DQj1Xb30XyM5j8DCJigZ5AhSKA3KI6C4RYSBL/CEk8xiCtp65R9rJhKSDlzy5fFtKKJqhTGwkYSYzt5EehPOYsUyFj2Q4GiWOCxbiwgzTQJbWfSOv2OD51FYF3mhWjut0r0StqbVxMvjh4OhQWIR1ReqLSKxHBs8qS5M45VOP6Yrbr3gVCtIbGFE29zoOyZhy/EqdF8p82DvhLwZJrLv5p4i/Yz+/ntM782g6MmAcZ9xnN/Ce+AXei0FqSpXdoP30V1JNFaOGt8qWWpDZqYihY+f4vCgcqYANK1gJpeiyBXkdBD50pCHjsqCxIsap6iXvsMF8ZFMhW6W68tqB5PUwsQOViuKrithlCrt4N5yQyq4PQTEwzDa2QkMFKnNFdRjHoEmc56KrP57IhxEg2eOzI5mK0g49Em3VMhJIiXxWqHAKavJMQBobXPb8vTO+Tq5OxsgolqZctdhPnOQ6cQUu3Y2rAoAQOTW/gXh4RZPMtaxTdFxHPiFvPWwg2EZob6WImzkKL1qAoy7FOBkZBsE0L+PNRlhp9g95Re7mEVYaHRChVnjNnIx9oBTrKPvmGTRdP3cbSOW+akCycMYyIRhFmUn9/4VappnWu/FtjqaS4mbXNr1tQVZsdRsb3pra7dfaVqPWqtc3djbLqnUbw3ZeV8sUdneRcz8IYXUR3/71+Icq3aXNWlTqPFEgXyOMXQcJtMJVNHbg+/GhAdPrgkTiH57EM6Yl82OmOrV+F8A6rKy0i8rvCpi+iE26EijKzx0BFm1v26fs62tPlwlUZ+06Dn83wCh62YlQ2A2iWALmWf31gD2ofbesenIBfT89LEvRT8Ti1N3mtw7dnoovhdypRadqV6wWHI54ZjqqqvhztNCaenIyJpFqI4rK3uyqyhbRLBsSUHzwgEqvOXZAGXigoWI77frTCPhLOjieeijh0SAL+Cm4hkwcQ8hOgj8UBwWKEo5IuDZKOILSDgopBdnAEWnWHZTWEqevPSdNIprMf4Wft5HQ584mo5a3w8ptPAzYs/E0PupjKh+W6bgEEdhzRarOuZa2NdMlnaaGZtVJ6ZIQ/Ci3S6772yqbQv8CVuPeFHqkzl/fsi3WxKJbmGBXTsrH3cO9vx6dJv0QEyvuRuaCMeHLvabD26pL+gLjjaeib/o+pbN+hVCYrSmYCs+DA++H3gqXU9OoPx+qRF2PCRC4Vehrq2g/ZRooZPfqKjwDS9a3l+PV8ta97EI/vQWdqa5trPOE4WUYoWgrixlPgivDvURL6XoiOJmEZJQGW2QkIzKJWxqXce3dgFSh32zfywT/CXEBWpvkKBMUV9ecFSm3nGUcKBZWFjsUhOBgMtmpV3QM3FaF8hsDISAd+nH2mFefHxtDYe84cZXnKtGNDa4R/EnGP9I1d87KvOFSMGWVEUhlmU6uU65oRyBMji60YJmwGj3deDUeXp03T1I8p8E2hJMabxdkM9nDIoB9TjeLO9rOxPtgLfbjz8dHZz8OTg9O++YRpVSTnjoy8bwcAIaC87uxv3TpPvQZaKeHZwj8Icbi9Vl6RVkJl1pp9saNE+MA2EnOlEpn5WAjRCBcQqdpOx2huvK574RX+dHDTLmoRpOxQDqcQdf2MncdUnJuNuKrQDO5w5MwuI1QV799qT5sXn7RKDr59lJ9WKqY3XZ9Q31YVWy0NkyXRxXtXqer59nbQB+2e7bdeXV9uMDaylJnqBS8H3J5MlFBeSkNT+3Sl1eXg2zoLscU0NHXTS9M5MaE3ScNL5rRJbHxoQ/TdeGf626oADRYfnhqk2q2xrcRU6PKnE5b64BLZkEjBNDVDtNIbUph00I0ZVVGM7OTtlKGyk8zhcv7dPgipDoFlQoXKKkk7oUovPq2aLGoHrx5NovY4MTo1pY6nypsfegjtLa2WHLaR2QdZBkyK446SmpO920jXMQNSM2jNRQZ8kUIzQhP/Ce3JaAXRXpWNBUQsA/ZABQTmtgsuSFB3NCMX0HpC51wlVzbiEdagaOFwiI8QzumEM1DkenfvNuZixOEdMcEHgiAz5hxFiNK0WBJF3zZ8sCxvJ6a7vxCuMSFvBVLfiCaHKVt0hq/l0RbjFndEMAFJ6ErthOGI07AvpCmm2n9G9/itB+RyXat3rE3TOuvKrYaG572ExWbzUbjeTbOdrPefv1IGpmul2Xu4kL0abGvqQUEDeni7IVOteE/ua76eixJEupGewAoxA2fO3o0saiIBEv8FIbkJNciFcblJK9kaEzRq6LIHFYYIQP/U0qYg6YYkGIYHuh+Xc+pN+6x0JdsKAr8h5kOgKyQA4ycthknGj2Tns6+7lURmpo47i0icouCs4uCsilmNzUXXYpr229lX0XxtFqXSbvFr4sja80yBZHAaYH7VAON8hA96NQ2bu66Fw1hXFEwgu5uksBlFegiDStYhDgQ7QNXzscS1iuMUeFYyWoqICaaOuL8pi+YE5L9mXPNU6Yyw2CVaxUcrV08rs5hYxZU4S6DL406nh+li2lAr3CuxVFyM56aJbcaF21ze5Nt/o8RNvY49XhKQN9zA/kepBaPhPBtGLq3EXn4hpFxT6I086XvZ5FOpATKYV27Vc9jXS4fUop2y7kqBtWSM9JjiuZClBwp33QUoHYt1VU6PwkKPh2s8cQFSY6mM/fXYemTce25q41Pvz1OFfWioxO+/y1ZjUKATdSuZkFQo4wUcJPAADLf6GGFKmxMj4XalbFQoFbJa8iE6AqKou9cXcELFJD7LA1piNDhzxMhW29MhlLh6RxeFEwlbCdOgodztClFCTLrTZGFKU2xmtkLmJWA8ryQzI7JSh26XizCfAF6M2PM8zFPzlRQ2INm61LBntGTVNJMTOjR57NChxI8b5gepdPPnz7tnvyMYV7Qk+UHjhuVy1j/XXoQh4LlljPQ8lYWAVSqiKvZc5nH0Tcj7r2YCJtpwT2JvtBiADYqobdkeGgK7inM+GySIzb+uZ9LB1/oEjK2QeJJEthTHjtR/mJHmBkxGxO89B2wZ/sLlruXE5EJrJ0QnZqAggBMH8eQXJt4n58IujBSm/aJMamwXPKSW3V3ZD4BPf6svz5Su87wPLns8LJSOK1j6m6EdS83vTiSHFfyNsibCvV6o3ql+yRLl5fFVdM7JR8X6uDP/X3houpzq+5djoYARuGKJn0WhLUaoqjC43T54a1a/Rgnpzg2VhtKrgm1oWQrM3ddI5pcnWtEvvvzx0+Pt5MVwHONiQLbVMBoLnMuHd+k1yagiplclaDiaNPkgAVXJVB+r53s2piFRC5jSZLOscDluUz9ktvOUnrYoUr/dYdorVnmsdhVAqWaIwBr3dRmAKv6UHhOHhtSfryUN+QSHBd2hQGdpiClgF3b3UUqWW0cgJppRRON1sbd4q/fKvY2070mMxVE3mYKa6LPk0Jl1ey+MFw2A4UhRq0570ZE3l3OFlG5SM4qCpa9f75nriifzRqz69G8MKA0c8wCtUJhYpQXedBJA8cI40xPs5PNlgJEyaIsQ3fVPAlCoEar0Rgsp25j5UKQmmPSNCZhTLOcZjQCLYZBqgaJQpFVEEifIBVc5NfQRpBqJFIUHEW49AjGNR75FFoFtXqrNBWsYbFdPSxWHJ8xDxCQTYCunVFp8qLY830259xF5UDzBepnHpSYKk7b5vVsMb0R55m8icLlR06I52LQM27YtdECLuE72jsBlAIeserDS9f1OT5JUjymt9Nq86+JpAFUDvX8hTDjgBBS5K6yxDcssOU2kKnr6AgMjyytT8oRllrt6YBxkhUBL0DAZHxEFaktXLcr4K34XdwISN5SdBN4eJpfKC3KY+DdJeZ/fa2sJPJMhl9H0oRC/m1oCk9Yi9Q/hI0AI4HpB8E16g4jPnVuvEDc5SzinoHhX+O53xea6c2byhrf4sCrtHy3OrVnmembLXvtFWdrzfR2z2rara79DNcrVW3SIdvXNdOnKCrwrM8W3hzPTfnBGOMRQFkUiTsN3bu6DpDn5yD/affk8ODwhz47+nQ8PPz8aXj2o8j9F3HcL6wl/HF1RdZn0LagatGC04UjSMmEyQYdujPv7+g9nNCxdQwlQMzCU2mO76Nt7iAWVfCybSxG+C9mQ/xmI2+OnjWPtPkF7BOZqfJoweffg5gKGw8Jc/I1yceCdlVUhmew57xtVLqB4Bh5WY4/DnZPB+wz/M8O96eD08HHn3OpWNbO+Is8NMn1KoxJFybCCju0i1Y0zZ51Xq5ZvZZdzTg7Kq8Rpqvd78meBKdd/73gVGF7OTh7RXA22ymcrwZikjQ9C6KdBXGzSGfbqmv8TrGoIr7Eyh8P/jzQBFlpO1IX81b1C301ceI3vsY3NUhZQGLkrS3Q2vHu/v7HwfB4/y/D/YPT3e/h86ej/cHH4enR55O9wXDvx8Hen2TebnRnGPf6iFhsnAVUB5NobDktacFrDNNwvXFcBtKEx++HQeiBZOAIRok2OG+yUrfIqjLLOYwa6POV/gLXWGjTaQvyfZEKQJkIE2lD3Dc8ROTagaEfbwOUNzY8HQG9w2Iihg+qaCLJw5WgoJmPn8eoNuK9q9qUCxXn+HgIdYf7gzOY6xO0HawtczLYk2U0xJmg3KQkpoymf31rLReoCJQLh+x64Q6BtmZ0+B6eAk4pHeuJrFIHEA8aFsZ6FrWmJpKJicTLg2/naJET4SLm5qJc97D8RnfCbpEULG9tXd9mjKICOWVIJt0LmTOQRMKYYS1CThiKcadrzwSsuV1SRb6XjQsO6HIDmHFhm6MIQuxO6uoUP/N3b1EOxdFnwuyIbAfqibzyE/o1Mnjhhn30hk3tKjFpIkpvjdYMSMYD7YaJrCFJ9JrcjJmfV7O4sIzqJwGMBsigHCcGZbxLwg9u8d6KyyL0SfMftu4NKv4F7yjWW670G/cZwv4F+vkNEiKaV102is6yvlyIJ7m4azfXhUGuFeJFxXp74/OEWLHb6T0nfpKqtnv1f4T4ydfMRgOvE56nV/vdstRkAMq5K//9Jq4xTqIn3tRNpLQGADnxJpS+UlzOE5DhBgj11TS1imQcpySyYYpgabUSVxNQxGBW/HmCW1D5u4IJnVGIsjyIboW5AyKYeo5kgKVBJu/Wk0nJf+LzGtHWWBz5N71upJbZSg4zMkgA1xdvz9F2LjGJPFIKbkljM2xDt07/aj/F7m+nySpemJnCLjKRP5ZS4mG7ub3WcL5JGgq70A1gPzcRhZ01r5Mt3dwYWg+ZdBHPtEQ2N8iQoCyRbYudYSRwLI2t6l4CpN599lnmPlqGuvU5/XSIiV0pPxFxx8wtA3ptRs5qMjDKzNJQBcNq6YoG7nN8JAyIVxiZ64m4ByPjucgSqtsOz9AeMhLONlg930/tGxTRfLz/IUKSJ+5kIROih6FjcywRJ4Fn+ACRLglPPtKEV7pbRQq5dNvNDPsZzK8A36bMd+ZXS7Rzk4DMyhyFj5APb/loGM1Y0+paGNVAacxBe/30vUyL+YMX/7gcVUQUHAatRZRIQsYvI/W74+RLnr5XEdukcMG45ok5W/R566TLJqeBu5Z2olcY0uM03ZAIo5AnisR46C6BFwpb5nWgzW+RvEHIL71GrbexxZQq1rubXmPS69dsq9Xp1ZvNTYUtWbVj269uMa2rLQX6jM65D+nKIdhgtOm/5u4sBMIfwxblDKqCGOCA5rRvJO3VLVbsAx9PHaZGXW+zWheGzIiPAJpz2FisXmPXV6wFO8FmexuJAW2rbpIQyi9NedjW0IaszWad9eTz4enZyee9s88ng/3EfnJ4dHawNxBaVT1vMFlqkFgJJBb1KwWI5CkphYaCTcqX/raMG46ME49prsS99bHKcIocO0KtCrqq0AH0dPWNkeCxCaQnwjJA1RCjyrxiDclWMhwm3A8B4892rDY3OPIIzOgnEelGfsyYj9C7Q2c1irhOksxNBJGJ/NUjyq0jzzIKM7y4I4WD/kzm+NwVaVQdiKwHMiXSYeFovMKMsyrTTqqSvpc4d5ecqqGjiPxuwUOPmBeLlleYNhhj3uTxEXnekjxyeOBSBc5lBFfymRpp2FFsQlYn/MCF3kIVg+fDsLYJDRMMg6khvxhPvXlQXsnFiT/RYgN5ASZJQ+JoDXITda6pKnL3Snch9kdRT/gCLwiccnmAiCp6+i1Yf6BtnSX9wD+gAUX0S+nhCESFBMVK1xy4lB8t+DgpipebARRDRRyPVzGsDFPnIRUXxlcLeiVfqPLiYbOU9KFqAKeZBEk3yN9mHmg9IRnYUo6l2vH01llJSgzwpqlRdtzaQ1gQPpcvS9ZipXYD4IraOLho7+62HwBYPRzJbAlDce8nD3NFFqsrQsKhD1wp1EBVU5SCWmpa9Qb8K4mFQqChhwmK0MhDWvr3IWivATZn/+H+D/8fAUu04w=='
original_bytes = zlib.decompress(base64.b64decode(ORIGINAL_B64))
assert hashlib.sha256(original_bytes).hexdigest() == '29717aec143ba39ed361c45fb7adfa620a3be9ef56b584b5db158f4bf1ca93c3'
original_nb = json.loads(original_bytes)
original_copy = LAB / 'original-class-notebook.ipynb'
original_copy.write_bytes(original_bytes)
print('Original preserved:', original_copy, '| cells:', len(original_nb['cells']))


### Inspect the original implementation
Set `ORIGINAL_CELL` to a cell number and rerun. The inspector prints source and saved text without executing them.

| Cell | Original example |
|---:|---|
| 5–6 | Docling conversion and final layout records |
| 9–12 | Text-only extraction, hidden-text disagreement, heavy scan |
| 15–18 | Typed Jev questions and hosted-session wiring |
| 21 | Tesseract crop/scale/mode experiment |
| 25 | Original imported acceptance policy call |
| 28–29 | PaddleOCR and fifth-reader policy evaluation |
| 31 | Unstructured text partitioning |

Unstructured returned one `NarrativeText` element for the short born-digital text in the original run. **Partitioning** produces document elements; it does not by itself establish which text is a supplier contact, or guarantee useful retrieval chunks. That distinction leads into Thursday.


In [ ]:
ORIGINAL_CELL = 15
c = original_nb['cells'][ORIGINAL_CELL]
print('ORIGINAL SOURCE — not executed:\n')
print(''.join(c['source']))
print('\nORIGINAL SAVED OUTPUT:\n')
for output in c.get('outputs', []):
    if 'text' in output:
        print(''.join(output['text']))


### Optional original-bundle setup
If your instructor supplies the trusted class ZIP, upload it through Colab’s Files sidebar and set its path below. The archive must contain `MANIFEST.json` and the original `fixtures/`, `research/`, and `code/` directories. The loader checks each manifest hash and rejects paths outside its destination. It does not import helper code or run hosted calls.

Leave the path empty to skip. A missing bundle should never block the 90-minute core.


In [ ]:
BUNDLE_ZIP = ''  # e.g. /content/class-bundle.zip, only when supplied by your instructor.
bundle_root = None
if BUNDLE_ZIP:
    import zipfile
    destination = (LAB / 'original-bundle').resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP) as archive:
        for entry in archive.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('Archive contains an unsafe path')
        archive.extractall(destination)
    manifests = list(destination.rglob('MANIFEST.json'))
    if len(manifests) != 1:
        raise ValueError('Expected exactly one MANIFEST.json')
    bundle_root = manifests[0].parent
    manifest = json.loads(manifests[0].read_text())
    for rel, expected_hash in manifest.items():
        path = (bundle_root / rel).resolve()
        if not path.is_relative_to(bundle_root.resolve()):
            raise ValueError('Unsafe manifest path')
        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash, rel
    print('Bundle files verified:', len(manifest))
else:
    print('No original bundle supplied; optional reproduction skipped.')


### Optional Docling reproduction on the original faint scan
Run only after verifying the bundle. Enabling this cell installs the Docling version named in the original notebook and may download model weights. It can take several minutes; Colab’s environment can differ from the instructor’s macOS environment. No exact score or output is promised.

Predict what you expect, run, then compare with the original **0.986** result. If the digits survive on your machine, record that outcome; do not rewrite it to match the class. The core tutorial remains unchanged.


In [ ]:
RUN_ORIGINAL_DOCLING = False
if RUN_ORIGINAL_DOCLING and bundle_root is not None:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'docling==2.142.1'], check=True)
    from docling.document_converter import DocumentConverter
    from PIL import Image
    source = bundle_root / 'fixtures' / 'fn-spot.jpg'
    display(Image.open(source))
    result = DocumentConverter().convert(str(source))
    print('LIVE original fixture on this runtime')
    print(result.document.export_to_markdown())
    print(result.confidence.pages)
else:
    print('Original Docling reproduction skipped; requires explicit opt-in and the original bundle.')


## Troubleshooting and instructor handoff

| Symptom | What to do |
|---|---|
| `NameError` in a core cell | Run setup, then the preceding section cells in order. A runtime restart clears variables. |
| Exercise says TRY AGAIN | This is expected until you complete the function. Read the failing input, edit, and rerun. |
| All cells ran but answers are empty | Execution is not completion. Return to prediction, coding, and exit-ticket prompts. |
| Optional OCR install fails | Skip the extension. No core assessment depends on package/model downloads. |
| Original manifest or fixture missing | Obtain the actual class bundle; do not substitute another scan and call it the same experiment. |
| A new run differs from the saved result | Preserve both and their environments. Interpret what your run shows. |

**For the instructor:** the core is planned for 90 minutes with pauses and student coding; optional parser reruns are extra. It uses embedded original outputs and new policy exercises. The original API wiring remains inspectable in the archive; no paid calls or credentials are activated. The new policy explicitly distinguishes not-found from proven absence and generic supplier contact from ordering contact. It is intentionally not a byte-for-byte implementation of the unavailable `accept_phone.py`.

**Suggested review rubric (10 points):** evidence/hypothesis distinction 2; controlled repair comparison 2; correct normalization and boundary handling 2; role-aware policy and provenance 2; clear message and next action 2. Test scores are supporting feedback, not a substitute for the student’s reasoning.

**Source:** supplied `itam-tue-document-processing.rendered.ipynb`. Recorded values and archived code are preserved from that file. No original fixture image is represented as available when it is not. Newly drawn schematics and practice data are explicitly labeled.
